# Lesson 12: Building Image Generation Applications

In this notebook (recommended **Runtime -> Change runtime type -> T4 GPU**), you will:
1. Wrap prompts in a child-safe visual metaprompt (`disallow_list`).
2. Generate images with open-source **Hugging Face `diffusers`** (or **Google AI Studio Imagen**).
3. Run **mask-based inpainting** on a self-contained `sunlit_lounge.png` + `mask.png` canvas.

In [ ]:
!pip install -q -U diffusers transformers accelerate pillow torch google-genai

In [ ]:
from PIL import Image, ImageDraw
import torch

# Create self-contained base image + alpha mask for inpainting
base = Image.new("RGB", (512, 512), (210, 225, 240))
draw = ImageDraw.Draw(base)
draw.rectangle([80, 260, 432, 460], fill=(40, 140, 200))  # indoor pool
base.save("sunlit_lounge.png")

mask = Image.new("L", (512, 512), 0)
mdraw = ImageDraw.Draw(mask)
mdraw.ellipse([200, 300, 312, 420], fill=255)  # white = region to inpaint
mask.save("mask.png")
print("Saved sunlit_lounge.png and mask.png | CUDA available:", torch.cuda.is_available())

In [ ]:
if torch.cuda.is_available():
    from diffusers import AutoPipelineForText2Image
    pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
    prompt = "Arc de Triomphe in Paris in evening light with a small child holding a teddy bear, storybook style, 16:9"
    img = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
    img.save("generated-image.png")
    print("Saved generated-image.png on T4 GPU!")
else:
    print("[CPU Mode] Switch Colab Runtime to T4 GPU to run Stable Diffusion Turbo inference.")